In [1]:
from model.raft.raft import RAFT

from types import SimpleNamespace


conf = {
    "cfg": 'config/raft/eval/sintel-S.json',
    "url": None,
    'path': 'weights/raft/Tartan-C-T-TSKH432x960-S.pth',
    "device": 'cpu',
    }
# conf = {}
# args = SimpleNamespace(**{**default_conf, **conf})

from config.parser import json_to_args
args =json_to_args('config/raft/eval/sintel-S.json')
args.path = 'weights/raft/Tartan-C-T-TSKH432x960-S.pth'
model = RAFT(args).eval()

# if args.path is not None:
#         model = RAFT(args)
#         # load_ckpt(model, args.path)



xFormers not available
xFormers not available


In [2]:
from utils.utils import load_ckpt, coords_grid, bilinear_sampler
load_ckpt(model, args.path)

In [3]:
from utils import frame_utils
import numpy as np
import torch, cv2

def preprocessing(image1_file, image2_file, is_test=True, size=(448,448)):
        if is_test:
            img1 = frame_utils.read_gen(image1_file)
            img2 = frame_utils.read_gen(image2_file)
            img1 = np.array(img1).astype(np.uint8)[..., :3]
            img2 = np.array(img2).astype(np.uint8)[..., :3]
            img1 = cv2.resize(img1, size) 
            img2 = cv2.resize(img2, size) 
            img1 = torch.from_numpy(img1).permute(2, 0, 1).float().unsqueeze(0)
            img2 = torch.from_numpy(img2).permute(2, 0, 1).float().unsqueeze(0)
            return img1, img2


image1_file = 'assets/frame_0016.png'
image2_file = 'assets/frame_0018.png'
image1, image2 = preprocessing(image1_file, image2_file)

In [4]:
image1.shape

torch.Size([1, 3, 448, 448])

In [5]:
# out = model.forward2(image1.cuda(), image2.cuda())
out = model(image1, image2, iters=args.iters, test_mode=True)

/home/rwang/py312/lib/python3.12/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [6]:
flow, info = out['flow'][-1], out['info'][-1]

In [7]:
from utils.flow_viz import flow_to_image

In [8]:
args.image_size

[432, 960]

In [9]:
flow_vis = flow_to_image(flow[0].cpu().permute(1, 2, 0).cpu().detach().numpy(), convert_to_bgr=True)
cv2.imwrite(f"./flow_raft_448.jpg", flow_vis)

True

In [9]:
args.iters

4

In [ ]:

import torch
import onnx
def export2onnx(model, dummy_input, out_file):
    with torch.no_grad():
        torch.onnx.export(model, dummy_input, 
                        out_file, 
                        verbose=True, opset_version=17)

    onnx_model = onnx.load(out_file)
    onnx.checker.check_model(onnx_model)

    try:
        import onnxsim
        onnx_model, check = onnxsim.simplify(onnx_model)
        assert check, 'assert check failed'
    except Exception as e:
        print(f'Simplify failure: {e}')
    onnx.save(onnx_model, out_file)
    print(f'ONNX export success, save into {out_file}')

data = {"image1": image1.cpu(), "image2": image2.cpu()}
model = model.eval().cpu()
model.forward = model.export
export2onnx(model, data, f'sea_raft_448.onnx')

/home/rwang/workspace/WAFT/model/raft/corr.py:117: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  return corr  / torch.sqrt(torch.tensor(dim).float())
/home/rwang/workspace/WAFT/model/raft/corr.py:117: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return corr  / torch.sqrt(torch.tensor(dim).float())


In [1]:
from model.raft.raft import RAFT

from types import SimpleNamespace


default_conf = {
    "cfg": 'config/raft/eval/sintel-S.json',
    "url": None,
    'path': None,
    "device": 'cpu',
    }
conf = {}
args = SimpleNamespace(**{**default_conf, **conf})

if args.path is not None:
        model = RAFT(args)
        # load_ckpt(model, args.path)



xFormers not available
xFormers not available


ModuleNotFoundError: No module named 'update'

In [ ]:
from model.raft.raft import RAFT

from types import SimpleNamespace


default_conf = {
    "cfg": 'config/raft/eval/sintel-S.json',
    "url": None,
    'path': None,
    "device": 'cpu',
    }
conf = {}
args = SimpleNamespace(**{**default_conf, **conf})

if args.path is not None:
        model = RAFT(args)
        # load_ckpt(model, args.path)



xFormers not available
xFormers not available


ModuleNotFoundError: No module named 'update'

In [10]:
import torch
import cv2
import os

@torch.no_grad()
def demo_data(model, image1, image2, valid=None, tiling=False):
    H, W = image1.shape[2:]
    output = model.calc_flow(image1, image2)
    for i in range(len(output['flow'])):
        flow= output['flow'][i]
        flow_vis = flow_to_image(flow[0].permute(1, 2, 0).cpu().numpy(), convert_to_bgr=True)
        cv2.imwrite(f"./flow_{i}.jpg", flow_vis)

# wrapped_model = wrapped_model.cuda()
demo_data(wrapped_model, image1.unsqueeze(0).cuda(), image2.unsqueeze(0).cuda())

/home/rwang/py312/lib/python3.12/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
